# 14 — Inter-annotator agreement

Reads the completed annotator workbooks, checks them for damage, measures
agreement, and exports every disagreement for adjudication.

Nothing is compared against the projection here. That comparison comes after
adjudication, in notebook 15, so that the gold set is settled before anyone
sees what the projection thought.

**Run from the repository root.** Kernel: `Python (tka)`. Under a minute.

## Cell 1: Load and validate

In [1]:
from pathlib import Path
import json, sys
from collections import Counter, defaultdict
import numpy as np
from openpyxl import load_workbook

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
ANN = ROOT / "annotation"
RES = ROOT / "results" / "ner"
RES.mkdir(parents=True, exist_ok=True)

books = sorted(ANN.glob("mizo_ner_annotation_*.xlsx"))
books = [b for b in books if not b.name.startswith("~$")]
print(f"found {len(books)} workbooks:")
for b in books:
    print(f"  {b.name}")
if len(books) < 2:
    sys.exit("Need at least two completed workbooks")

key = json.load(open(ANN / "gold_sample_key.json", encoding="utf-8"))
key_by_id = {k["key_id"]: k for k in key}
print(f"\nkey: {len(key)} sentences, "
      f"{sum(len(k['tokens']) for k in key):,} tokens")

TYPES = ["PERSON","GPE","ORG","NORP","LOC","LANGUAGE",
         "WORK_OF_ART","FAC","PRODUCT","EVENT","LAW"]
VALID = {"O"} | {f"{p}-{e}" for e in TYPES for p in ("B","I")}

def read_book(path):
    ws = load_workbook(path, data_only=True)["Annotate"]
    rows = {}
    for r in ws.iter_rows(min_row=2, values_only=True):
        if r[0] is None:
            continue
        sent, tok, word, tag = int(r[0]), int(r[2]), r[3], r[4]
        rows[(sent, tok)] = (word, (tag or "").strip())
    return rows

ann = {}
for b in books:
    name = b.stem.replace("mizo_ner_annotation_", "")
    ann[name] = read_book(b)
    print(f"  {name}: {len(ann[name]):,} token rows")

found 2 workbooks:
  mizo_ner_annotation_A.xlsx
  mizo_ner_annotation_B.xlsx

key: 300 sentences, 3,644 tokens
  A: 3,644 token rows
  B: 3,644 token rows


## Cell 2: Integrity checks

Annotators work in Excel, so rows get deleted, tags get typed by hand, and
files get saved in odd states. Every one of these would corrupt the measurement
silently.

In [2]:
names = sorted(ann)
expected = {(k["key_id"], j + 1): tok
            for k in key for j, tok in enumerate(k["tokens"])}
print(f"expected {len(expected):,} token rows\n")

problems = 0
for n in names:
    rows = ann[n]
    missing = set(expected) - set(rows)
    extra   = set(rows) - set(expected)
    mismatch = [k for k in set(rows) & set(expected)
                if rows[k][0] != expected[k]]
    blank = [k for k, (_, t) in rows.items() if not t]
    invalid = sorted({t for _, t in rows.values() if t and t not in VALID})

    print(f"--- annotator {n} ---")
    print(f"  missing rows      : {len(missing):,}")
    print(f"  unexpected rows   : {len(extra):,}")
    print(f"  word text altered : {len(mismatch):,}")
    print(f"  blank tags        : {len(blank):,}")
    print(f"  invalid tags      : {invalid if invalid else 'none'}")
    problems += len(missing) + len(extra) + len(mismatch) + len(blank) + len(invalid)
    if missing:
        print(f"    e.g. {sorted(missing)[:5]}")
    if invalid:
        print("    fix these in the workbook and re-run")

if problems:
    print(f"\n{problems} problems found. Resolve before trusting the numbers below.")
else:
    print("\nBoth workbooks are intact.")

expected 3,644 token rows

--- annotator A ---
  missing rows      : 0
  unexpected rows   : 0
  word text altered : 2
  blank tags        : 0
  invalid tags      : none
--- annotator B ---
  missing rows      : 0
  unexpected rows   : 0
  word text altered : 1
  blank tags        : 0
  invalid tags      : none

3 problems found. Resolve before trusting the numbers below.


## Cell 3: BIO well-formedness

An `I-TYPE` that does not follow a `B-TYPE` of the same type is a slip, not a
judgment. We repair these silently for the agreement calculation and report how
many there were, since a high count suggests the annotator misunderstood the
scheme.

In [3]:
def sentence_tags(rows, key_id, n_tok):
    return [rows.get((key_id, j + 1), ("", "O"))[1] or "O" for j in range(n_tok)]

def fix_bio(tags):
    out, prev, fixes = [], "O", 0
    for t in tags:
        if t.startswith("I-"):
            typ = t[2:]
            if not (prev == f"B-{typ}" or prev == f"I-{typ}"):
                t = f"B-{typ}"; fixes += 1
        out.append(t); prev = t
    return out, fixes

tagged = {}
for n in names:
    total_fix = 0
    seqs = []
    for k in key:
        raw = sentence_tags(ann[n], k["key_id"], len(k["tokens"]))
        fixed, f = fix_bio(raw)
        total_fix += f
        seqs.append(fixed)
    tagged[n] = seqs
    nent = sum(1 for s in seqs for t in s if t.startswith("B-"))
    print(f"  {n}: {nent:,} entities marked, {total_fix} stray I- tags repaired")

  A: 399 entities marked, 3 stray I- tags repaired
  B: 394 entities marked, 3 stray I- tags repaired


## Cell 4: Agreement

Two measures, because they answer different questions. Cohen's $\kappa$ over
token labels is the conventional figure, but most tokens are `O`, so it is
sensitive to that prevalence. Entity-level $F_1$ between the two annotators
treats one as reference and the other as prediction, and reflects agreement on
the spans that actually matter.

In [4]:
from sklearn.metrics import cohen_kappa_score
from seqeval.metrics import f1_score, precision_score, recall_score

a, b = names[0], names[1]
flat_a = [t for s in tagged[a] for t in s]
flat_b = [t for s in tagged[b] for t in s]

raw_agree = np.mean([x == y for x, y in zip(flat_a, flat_b)])
kappa_full = cohen_kappa_score(flat_a, flat_b)

bin_a = ["ENT" if t != "O" else "O" for t in flat_a]
bin_b = ["ENT" if t != "O" else "O" for t in flat_b]
kappa_bin = cohen_kappa_score(bin_a, bin_b)

ent_f1 = f1_score(tagged[a], tagged[b])

print(f"tokens compared            : {len(flat_a):,}")
print(f"raw token agreement        : {raw_agree*100:.2f}%")
print(f"Cohen's kappa (23 labels)  : {kappa_full:.4f}")
print(f"Cohen's kappa (entity/not) : {kappa_bin:.4f}")
print(f"entity-level F1 ({a} vs {b}) : {ent_f1:.4f}")
print(f"""
Interpretation of kappa: >0.80 very good, 0.60-0.80 substantial,
0.40-0.60 moderate. Ghosh et al. report 0.90 for their Mizo NER data.
""")

# where do they disagree, by type?
print(f"{'Type':<14}{a+' only':>10}{b+' only':>10}{'both':>8}")
print("-" * 42)
for t in TYPES:
    sa = {(i, j) for i, s in enumerate(tagged[a]) for j, x in enumerate(s) if x == f"B-{t}"}
    sb = {(i, j) for i, s in enumerate(tagged[b]) for j, x in enumerate(s) if x == f"B-{t}"}
    if sa or sb:
        print(f"{t:<14}{len(sa-sb):>10}{len(sb-sa):>10}{len(sa&sb):>8}")

tokens compared            : 3,644
raw token agreement        : 95.09%
Cohen's kappa (23 labels)  : 0.7979
Cohen's kappa (entity/not) : 0.8856
entity-level F1 (A vs B) : 0.7087

Interpretation of kappa: >0.80 very good, 0.60-0.80 substantial,
0.40-0.60 moderate. Ghosh et al. report 0.90 for their Mizo NER data.

Type              A only    B only    both
------------------------------------------
PERSON                28        19     213
GPE                   19        26      18
ORG                   21        17      27
NORP                  11         4       5
LOC                   17        26      33
LANGUAGE               1         3       3
FAC                    1         0       0
EVENT                  2         0       0


## Cell 5: Export disagreements for adjudication

In [5]:
from openpyxl import Workbook
from openpyxl.worksheet.datavalidation import DataValidation
from openpyxl.styles import Font, PatternFill, Alignment

HDR  = PatternFill("solid", fgColor="DDDDDD")
DIFF = PatternFill("solid", fgColor="FFF2CC")
SENT = PatternFill("solid", fgColor="F2F7FF")

wb = Workbook(); ws = wb.active; ws.title = "Adjudicate"
ws.append(["Sent", "Sentence", "Tok", "Word",
           f"Tag {a}", f"Tag {b}", "FINAL", "Reason"])
for c in ws[1]: c.font = Font(bold=True); c.fill = HDR
ws.freeze_panes = "A2"

r = 2
n_disagree_sent = 0
for i, k in enumerate(key):
    ta, tb = tagged[a][i], tagged[b][i]
    if ta == tb:
        continue
    n_disagree_sent += 1
    text = " ".join(k["tokens"])
    for j, (tok, x, y) in enumerate(zip(k["tokens"], ta, tb)):
        ws.cell(row=r, column=1, value=k["key_id"])
        if j == 0:
            c = ws.cell(row=r, column=2, value=text)
            c.fill = SENT; c.alignment = Alignment(wrap_text=True, vertical="top")
        ws.cell(row=r, column=3, value=j + 1)
        ws.cell(row=r, column=4, value=tok)
        ca = ws.cell(row=r, column=5, value=x)
        cb = ws.cell(row=r, column=6, value=y)
        cf = ws.cell(row=r, column=7, value="")
        if x != y:
            ca.fill = DIFF; cb.fill = DIFF; cf.fill = DIFF
            cf.value = x            # pre-fill with A; adjudicator overrides
        else:
            cf.value = x
        r += 1
    r += 1

dv = DataValidation(type="list",
                    formula1='"' + ",".join(["O"] + [f"{p}-{e}" for e in TYPES
                                                     for p in ("B","I")]) + '"',
                    allow_blank=False, showDropDown=False)
ws.add_data_validation(dv); dv.add(f"G2:G{r}")
for col, w in (("A",6),("B",56),("C",6),("D",22),("E",14),("F",14),("G",14),("H",30)):
    ws.column_dimensions[col].width = w

out = ANN / "adjudication.xlsx"
wb.save(out)
print(f"-> {out.relative_to(ROOT)}")
print(f"{n_disagree_sent} of {len(key)} sentences contain at least one disagreement")
print(f"{r-2:,} rows written (whole sentences, so context is visible)")
print("""
Highlighted cells are the disagreements. The FINAL column is pre-filled with
annotator """ + a + """'s tag; change it where """ + b + """ is right, or enter a
third answer where both are wrong. Work through it together if you can.
""")

json.dump({"annotators": names, "sentences": len(key), "tokens": len(flat_a),
           "raw_agreement_pct": round(float(raw_agree)*100, 2),
           "cohen_kappa_labels": round(float(kappa_full), 4),
           "cohen_kappa_binary": round(float(kappa_bin), 4),
           "entity_f1_between_annotators": round(float(ent_f1), 4),
           "sentences_with_disagreement": n_disagree_sent},
          open(RES / "annotator_agreement.json", "w"), indent=2)
print(f"-> results/ner/annotator_agreement.json")

-> annotation\adjudication.xlsx
110 of 300 sentences contain at least one disagreement
1,712 rows written (whole sentences, so context is visible)

Highlighted cells are the disagreements. The FINAL column is pre-filled with
annotator A's tag; change it where B is right, or enter a
third answer where both are wrong. Work through it together if you can.

-> results/ner/annotator_agreement.json


In [6]:
from collections import Counter
cross = Counter()
for sa, sb in zip(tagged[a], tagged[b]):
    for x, y in zip(sa, sb):
        if x != y:
            tx = x[2:] if x != "O" else "O"
            ty = y[2:] if y != "O" else "O"
            if tx != ty:
                cross[(tx, ty)] += 1
print(f"{a} said / {b} said        count")
print("-" * 38)
for (x, y), n in cross.most_common(12):
    print(f"  {x:<14} {y:<14}{n:>5}")

same_span = sum(n for (x, y), n in cross.items() if x != "O" and y != "O")
print(f"\nboth marked an entity, disagreed on type: {same_span}")
print(f"one marked an entity, other said O      : "
      f"{sum(n for (x,y),n in cross.items() if x=='O' or y=='O')}")

A said / B said        count
--------------------------------------
  O              GPE              16
  GPE            LOC              14
  PERSON         O                14
  ORG            GPE              13
  O              ORG              12
  ORG            O                12
  O              LOC              10
  LOC            O                 8
  O              PERSON            7
  NORP           O                 6
  LOC            GPE               5
  LOC            PERSON            5

both marked an entity, disagreed on type: 74
one marked an entity, other said O      : 96


In [7]:
def merge_loc(tags):
    out = []
    for t in tags:
        if t.endswith("-GPE") or t.endswith("-LOC"):
            out.append(t[:2] + "LOCATION")
        else:
            out.append(t)
    return out

ma = [merge_loc(s) for s in tagged[a]]
mb = [merge_loc(s) for s in tagged[b]]
fa = [t for s in ma for t in s]; fb = [t for s in mb for t in s]

print(f"kappa, GPE and LOC separate : {cohen_kappa_score(flat_a, flat_b):.4f}")
print(f"kappa, merged to LOCATION   : {cohen_kappa_score(fa, fb):.4f}")
print(f"entity F1, separate         : {f1_score(tagged[a], tagged[b]):.4f}")
print(f"entity F1, merged           : {f1_score(ma, mb):.4f}")

kappa, GPE and LOC separate : 0.7979
kappa, merged to LOCATION   : 0.8180
entity F1, separate         : 0.7087
entity F1, merged           : 0.7440
